In [1]:
import os
os.chdir("../../")
print("Current working directory:", os.getcwd())
import csv
import nest_asyncio
nest_asyncio.apply()
from llama_index.core import (
    Settings,
    Document,
    VectorStoreIndex,
    StorageContext,
    load_index_from_storage,
)
from llama_index.vector_stores.qdrant import QdrantVectorStore
from qdrant_client import QdrantClient, AsyncQdrantClient
from llama_index.embeddings.text_embeddings_inference import TextEmbeddingsInference
from llama_index.llms.openai_like import OpenAILike
from IPython.display import display, Markdown
# By pass SSL certification
import httpx

# Apply the monkey patch
# Apply the monkey patch
from chainlit_app.patches import patch

patch.apply_patch()


# Trace llamaindex
from openinference.instrumentation.llama_index import LlamaIndexInstrumentor
from phoenix.otel import register

Current working directory: /Users/chirawatchitpakdee/python_project/ai-chatbot


/Users/chirawatchitpakdee/python_project/ai-chatbot/.conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
2025-01-22 09:50:42,161 - INFO - Patched TextEmbeddingsInference._call_api with custom synchronous API handling
2025-01-22 09:50:42,162 - INFO - Patched TextEmbeddingsInference._acall_api with custom asynchronous API handling


In [2]:
tracer_provider = register(
    project_name="test-indexing",
    endpoint="http://localhost:3000/v1/traces",
)

LlamaIndexInstrumentor().instrument(
    skip_dep_check=True, tracer_provider=tracer_provider
)

🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: test-indexing
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: http://localhost:3000/v1/traces
|  Transport: HTTP
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [3]:
# Create an httpx client with SSL verification disabled
http_client = httpx.Client(verify=False)
# Create an httpx AsyncClient with SSL verification using certifi
async_http_client = httpx.AsyncClient(verify=False)

# Set up LLM model
Settings.llm = OpenAILike(
    model="default",
    api_base="https://api.cpxis.global.lotuss.org/llm/v1",
    api_key="automation.lotuss.Zb71t4pjNR3rty3uI8os9jwxaJmU8h",
    is_chat_model=True,
    is_function_calling_model=False,
    temperature=0.2,
    http_client=http_client,
    async_http_client=async_http_client,
    # max_tokens=4096,
)

# Initialize the embedding settings
embed_model = TextEmbeddingsInference(
    model_name="BAAI/bge-m3",
    base_url=f"https://api.cpxis.global.lotuss.org/embedding/BAAI/bge-m3",
    auth_token=f"Bearer automation.lotuss.Zb71t4pjNR3rty3uI8os9jwxaJmU8h",
    timeout=60,
    embed_batch_size=10,
)

Settings.embed_model = embed_model

In [4]:
from llama_index.core.vector_stores import VectorStoreQueryResult
from llama_index.core.vector_stores import VectorStoreQueryResult


def relative_score_fusion(
    dense_result: VectorStoreQueryResult,
    sparse_result: VectorStoreQueryResult,
    alpha: float = 0.5,  # passed in from the query engine
    top_k: int = 2,  # passed in from the query engine i.e. similarity_top_k
) -> VectorStoreQueryResult:
    """
    Fuse dense and sparse results using relative score fusion.
    """
    print("Using Qdrant Hybrid Search: alpha:", alpha)
    # sanity check
    assert dense_result.nodes is not None
    assert dense_result.similarities is not None
    assert sparse_result.nodes is not None
    assert sparse_result.similarities is not None

    # deconstruct results
    sparse_result_tuples = list(
        zip(sparse_result.similarities, sparse_result.nodes)
    )
    sparse_result_tuples.sort(key=lambda x: x[0], reverse=True)

    dense_result_tuples = list(
        zip(dense_result.similarities, dense_result.nodes)
    )
    dense_result_tuples.sort(key=lambda x: x[0], reverse=True)

    # track nodes in both results
    all_nodes_dict = {x.node_id: x for x in dense_result.nodes}
    for node in sparse_result.nodes:
        if node.node_id not in all_nodes_dict:
            all_nodes_dict[node.node_id] = node

    # normalize sparse similarities from 0 to 1
    sparse_similarities = [x[0] for x in sparse_result_tuples]
    max_sparse_sim = max(sparse_similarities)
    min_sparse_sim = min(sparse_similarities)
    sparse_similarities = [
        (x - min_sparse_sim) / (max_sparse_sim - min_sparse_sim)
        for x in sparse_similarities
    ]
    sparse_per_node = {
        sparse_result_tuples[i][1].node_id: x
        for i, x in enumerate(sparse_similarities)
    }

    # normalize dense similarities from 0 to 1
    dense_similarities = [x[0] for x in dense_result_tuples]
    max_dense_sim = max(dense_similarities)
    min_dense_sim = min(dense_similarities)
    dense_similarities = [
        (x - min_dense_sim) / (max_dense_sim - min_dense_sim)
        for x in dense_similarities
    ]
    dense_per_node = {
        dense_result_tuples[i][1].node_id: x
        for i, x in enumerate(dense_similarities)
    }

    # fuse the scores
    fused_similarities = []
    for node_id in all_nodes_dict:
        sparse_sim = sparse_per_node.get(node_id, 0)
        dense_sim = dense_per_node.get(node_id, 0)
        fused_sim = alpha * (sparse_sim + dense_sim)
        fused_similarities.append((fused_sim, all_nodes_dict[node_id]))

    fused_similarities.sort(key=lambda x: x[0], reverse=True)
    fused_similarities = fused_similarities[:top_k]

    # create final response object
    return VectorStoreQueryResult(
        nodes=[x[1] for x in fused_similarities],
        similarities=[x[0] for x in fused_similarities],
        ids=[x[1].node_id for x in fused_similarities],
    )

In [5]:
api_key = "QdrantVAsfhF8nGPtyleJKVkt2TBI2bqQ4bSjgnajNtOLwE2Y9YWxnZFrItRBE53"
# creates a persistant index to disk
client = QdrantClient(url="http://localhost:6334", api_key=api_key,  prefer_grpc=True)
aclient = AsyncQdrantClient(url="http://localhost:6334", api_key=api_key, prefer_grpc=True)

# create our vector store with hybrid indexing enabled
# batch_size controls how many nodes are encoded with sparse vectors at once
vector_store = QdrantVectorStore(
    "vector_data",
    client=client,
    aclient=aclient,
    enable_hybrid=True,
    batch_size=20,
    prefer_grpc=True,
    hybrid_fusion_fn=relative_score_fusion,
)


/Users/chirawatchitpakdee/python_project/ai-chatbot/.conda/lib/python3.11/site-packages/qdrant_client/qdrant_remote.py:130: UserWarning: Api key is used with an insecure connection.
  warnings.warn("Api key is used with an insecure connection.")
/Users/chirawatchitpakdee/python_project/ai-chatbot/.conda/lib/python3.11/site-packages/qdrant_client/async_qdrant_remote.py:117: UserWarning: Api key is used with an insecure connection.
  warnings.warn("Api key is used with an insecure connection.")
2025-01-22 09:50:42,330 - WARNING - Both client and aclient are provided. If using `:memory:` mode, the data between clients is not synced.


In [6]:
# Define chat profiles and their specific settings
system_prompt = """You are a female customer service officer with over 10 years of work experience that can converse both in English and Thai. Your role is to provide professional customer service by answering questions and offering additional advice related to Lotus's inquiries, ensuring that customers are impressed and have a positive experience every day at Lotus's.\n
If the question is not related to Lotus's, respond with: For Thai conversation: ขออภัยค่ะ หากต้องการทราบข้อมูลเพิ่มเติม สามารถติดต่อศูนย์บริการลูกค้าโลตัสที่หมายเลข 1430 ได้เลยค่ะ, For English conversation: Apologies. , if you need more information, you can contact the Lotus Customer Service Center at 1430.\n
If the question is related to My Lotus's but cannot be answered, respond with: For Thai conversation: กรุณาสอบถามเพิ่มเติมที่ 1430 ทุกวันตั้งแต่เวลา 9:00 น. ถึง 23:00 น., For English conversation: Please contact 1430 for further inquiries, available every day from 9:00 AM to 11:00 PM.\n
If the question is related to Lotus's Shop Online but cannot be answered, respond with: For Thai conversation: กรุณาสอบถามเพิ่มเติมที่ 1430 กด 2 ทุกวันตั้งแต่เวลา 9:00 น. ถึง 23:00 น., For English conversation: Please contact 1430, press 2, for further inquiries, available every day from 9:00 AM to 11:00 PM.\n
Answer the user's questions using the provided context, sticking to the facts. Do not draw conclusions on your own.\n
Please answer the questions correctly, in a friendly and slightly playful manner, while being polite, complete, and clear.\n
If the user asks in Thai, please answer in Thai.\n
End every Thai conversation with the following sentence: หากมีคำถามเพิ่มเติม สามารถสอบถามน้องบัวได้เลยนะคะ ขอบคุณที่ใช้บริการค่ะ\n
If the user asks in English, please answer in English.\n
End every English conversation with the following sentence: If you have any further questions, feel free to ask Nong Bua. Thank you for using our service!.\n
Please carefully indentify user language, think twice use the same language as user question.\n
Please make sure to answer only in Thai or English language, Not other language.\n
You are female customer service officer, Please use คะ or ค่ะ not ครับ when answer in thai language.\n
If relevant documents for the context have emoji, Please answer with emoji.\n
Do NOT rely on prior knowledge.\n"""

In [7]:
loaded_index = VectorStoreIndex.from_vector_store(vector_store, use_async=True)

### Condense Plus Context - Hybrid Search

In [ ]:
from llama_index.core.storage.chat_store import SimpleChatStore
from llama_index.core.memory import ChatMemoryBuffer

chat_store = SimpleChatStore()

chat_memory = ChatMemoryBuffer.from_defaults(
    token_limit=8192,
    chat_store=chat_store,
)

In [9]:
chat_engine = loaded_index.as_chat_engine(
    chat_mode="condense_plus_context",
    memory=chat_memory,
    similarity_top_k=5,
    sparse_top_k=12,
    alpha=0.5,
    system_prompt=system_prompt,
    vector_store_query_mode="hybrid",
    verbose=True,
)

In [10]:
user_query = "สอบถามขั้นตอนการเข้าใช้งานแอปพลิเคชั่น"
response = await chat_engine.achat(user_query)
display(Markdown(str(response)))

2025-01-22 09:50:45,804 - INFO - Condensed question: สอบถามขั้นตอนการเข้าใช้งานแอปพลิเคชั่น
2025-01-22 09:50:45,854 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/embedding/BAAI/bge-m3/embed "HTTP/1.1 200 OK"


Condensed question: สอบถามขั้นตอนการเข้าใช้งานแอปพลิเคชั่น
Using Qdrant Hybrid Search: alpha: 0.5


2025-01-22 09:50:51,242 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/llm/v1/chat/completions "HTTP/1.1 200 OK"


ขั้นตอนการเข้าใช้งานแอปพลิเคชัน Lotus's SMART App คือ 
1. ดาวน์โหลดแอปพลิเคชัน Lotus's SMART App จาก App Store หรือ Google Play Store
2. เปิดแอปพลิเคชัน Lotus's SMART App 
3. กดที่ "ลงชื่อเข้าใช้"
4. ใส่หมายเลขโทรศัพท์ที่สมัครสมาชิก My Lotus's 
5. ใส่รหัสผ่าน 
6. กดที่ "เข้าใช้งาน" ค่ะ หากมีคำถามเพิ่มเติม สามารถสอบถามน้องบัวได้เลยนะคะ ขอบคุณที่ใช้บริการค่ะ

In [11]:
user_query = "ถ้าต้องการสะสมคะแนนโลตัสต้องทำไงคะ"
response = await chat_engine.achat(user_query)
display(Markdown(str(response)))

2025-01-22 09:50:52,184 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/llm/v1/chat/completions "HTTP/1.1 200 OK"
2025-01-22 09:50:52,215 - INFO - Condensed question: วิธีการสะสมคะแนนโลตัสเมื่อใช้งานแอปพลิเคชัน Lotus's SMART App คืออะไรคะ
2025-01-22 09:50:52,253 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/embedding/BAAI/bge-m3/embed "HTTP/1.1 200 OK"


Condensed question: วิธีการสะสมคะแนนโลตัสเมื่อใช้งานแอปพลิเคชัน Lotus's SMART App คืออะไรคะ
Using Qdrant Hybrid Search: alpha: 0.5


2025-01-22 09:51:04,205 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/llm/v1/chat/completions "HTTP/1.1 200 OK"
2025-01-22 09:51:18,194 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/llm/v1/chat/completions "HTTP/1.1 200 OK"


คะแนนโลตัสหรือโลตัสคอยน์ คือ คะแนนสะสมพิเศษที่คุณลูกค้าจะได้รับเมื่อซื้อสินค้าที่ร่วมรายการกับโลตัส ซึ่งสามารถสะสมและใช้แลกเป็นสิทธิพิเศษได้ โดยคุณลูกค้าสามารถใช้โลตัสคอยน์แทนเงินสดเพื่อรับส่วนลดในการซื้อสินค้าที่ร่วมรายการที่โลตัสทุกสาขา หรือ โลตัสช้อปออนไลน์ นอกจากนี้ คุณลูกค้ายังสามารถใช้โลตัสคอยน์แลกสิทธิพิเศษ เช่น พริวิเลจดีล คูปองส่วนลด คูปองเงินสด และข้อเสนอจากพาร์ทเนอร์ต่าง ๆ ผ่าน โลตัส สมาร์ท แอป 

การสะสมคะแนนโลตัสคอยน์ คือ 
- ซื้อสินค้าที่ร่วมรายการกับโลตัส 
- สำหรับทุกๆ 50 บาทที่ซื้อ จะได้รับ 0.5% เป็นโลตัสคอยน์ หรือ 5 โลตัสคอยน์ สำหรับการซื้อ 1,000 บาท ค่ะ 

การตรวจสอบยอดโลตัสคอยน์:
คุณลูกค้าสามารถเช็คยอดสะสมได้ตลอดเวลาผ่านช่องทางดังนี้ค่ะ
1. Lotus Smart App: เข้าเมนู "มายโลตัส"
2. ใบเสร็จรับเงิน: ดูได้ที่ท้ายใบเสร็จ
3. Line Official: กดเมนู "สมาชิกมายโลตัส"
4. ศูนย์บริการลูกค้า: โทร 1430 กด 1 ได้ทุกวันตั้งแต่เวลา 9:00 – 23:00 น. ค่ะ

หมายเหตุ:
โลตัสคอยน์ 1 เหรียญ มีมูลค่าเท่ากับ 1 บาท ใช้เป็นส่วนลดแทนเงินสดได้เลยค่ะ 😊 หากมีคำถามเพิ่มเติม สามารถสอบถามน้องบัวได้เลยนะคะ ขอบคุณที่ใช้บริการค่ะ

In [12]:
user_query = "ขอที่อยู่ของ โลตัส โกเฟรช โพไร่หวาน"
response = await chat_engine.achat(user_query)
display(Markdown(str(response)))

2025-01-22 09:51:19,228 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/llm/v1/chat/completions "HTTP/1.1 200 OK"
2025-01-22 09:51:19,255 - INFO - Condensed question: ที่อยู่ของ โลตัส โกเฟรช โพไร่หวาน คืออะไรคะ
2025-01-22 09:51:19,387 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/embedding/BAAI/bge-m3/embed "HTTP/1.1 200 OK"


Condensed question: ที่อยู่ของ โลตัส โกเฟรช โพไร่หวาน คืออะไรคะ
Using Qdrant Hybrid Search: alpha: 0.5


2025-01-22 09:51:24,599 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/llm/v1/chat/completions "HTTP/1.1 200 OK"
2025-01-22 09:51:29,649 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/llm/v1/chat/completions "HTTP/1.1 200 OK"


ที่อยู่ของ โลตัส โกเฟรช โพไร่หวาน เพชรบุรี คือ 
เลขที่ 223 หมู่ 2 ต.โพไร่หวาน อ.เมืองเพชรบุรี จ.เพชรบุรี 76000 
โทรศัพท์: 064-587-4193 ค่ะ 
หรือคุณลูกค้าสามารถเข้าไปดูที่ Google Map ได้ที่ https://www.google.com/maps/place/13.079519,99.96638 ค่ะ หากมีคำถามเพิ่มเติม สามารถสอบถามน้องบัวได้เลยนะคะ ขอบคุณที่ใช้บริการค่ะ

In [13]:
user_query = "ข้อข้อมูลสาขาในจังหวัดบุรีรัมย์"
response = await chat_engine.achat(user_query)
display(Markdown(str(response)))

2025-01-22 09:51:30,484 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/llm/v1/chat/completions "HTTP/1.1 200 OK"
2025-01-22 09:51:30,506 - INFO - Condensed question: ขอข้อมูลเกี่ยวกับสาขาโลตัสในจังหวัดบุรีรัมย์


Condensed question: ขอข้อมูลเกี่ยวกับสาขาโลตัสในจังหวัดบุรีรัมย์


2025-01-22 09:51:30,714 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/embedding/BAAI/bge-m3/embed "HTTP/1.1 200 OK"


Using Qdrant Hybrid Search: alpha: 0.5


2025-01-22 09:51:36,626 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/llm/v1/chat/completions "HTTP/1.1 200 OK"
2025-01-22 09:51:42,493 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/llm/v1/chat/completions "HTTP/1.1 200 OK"
2025-01-22 09:51:52,008 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/llm/v1/chat/completions "HTTP/1.1 200 OK"
2025-01-22 09:51:59,153 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/llm/v1/chat/completions "HTTP/1.1 200 OK"


ข้อมูลสาขาโลตัสในจังหวัดบุรีรัมย์ คือ 
โลตัส โกเฟรช นางรอง 2 บุรีรัมย์ 
ที่อยู่: เลขที่ 589/2,589/3 ถนนประจันตเขต ตำบลนางรอง อำเภอนางรอง จังหวัดบุรีรัมย์ 31110 
โทรศัพท์: 062-605-8841 
เวลาเปิด-ปิด: 06:00-23:00 น. 
หรือคุณลูกค้าสามารถเข้าไปดูที่ Google Map ได้ที่ https://www.google.com/maps/place/14.636646,102.793516 ค่ะ 
นอกจากนี้ยังมี โลตัส สาขาใหญ่ ในจังหวัดบุรีรัมย์ แต่ข้อมูลที่มีอยู่ไม่เพียงพอ ค่ะ หากมีคำถามเพิ่มเติม สามารถสอบถามน้องบัวได้เลยนะคะ ขอบคุณที่ใช้บริการค่ะ

### Condense Plus Context - Regular Dense Search

In [14]:
from llama_index.core.storage.chat_store import SimpleChatStore
from llama_index.core.memory import ChatMemoryBuffer

chat_store = SimpleChatStore()

chat_memory = ChatMemoryBuffer.from_defaults(
    token_limit=8192,
    chat_store=chat_store,
)

In [15]:
chat_engine = loaded_index.as_chat_engine(
    chat_mode="condense_plus_context",
    memory=chat_memory,
    similarity_top_k=5,
    sparse_top_k=12,
    # alpha=0.5,
    system_prompt=system_prompt,
    # vector_store_query_mode="hybrid",
    verbose=True,
)

In [16]:
user_query = "สอบถามขั้นตอนการเข้าใช้งานแอปพลิเคชั่น"
response = await chat_engine.achat(user_query)
display(Markdown(str(response)))

2025-01-22 09:56:25,575 - INFO - Condensed question: สอบถามขั้นตอนการเข้าใช้งานแอปพลิเคชั่น
2025-01-22 09:56:25,635 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/embedding/BAAI/bge-m3/embed "HTTP/1.1 200 OK"


Condensed question: สอบถามขั้นตอนการเข้าใช้งานแอปพลิเคชั่น


2025-01-22 09:56:30,528 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/llm/v1/chat/completions "HTTP/1.1 200 OK"


ขั้นตอนการเข้าใช้งานแอปพลิเคชัน Lotus's SMART App คือ 
1. ดาวน์โหลดแอปพลิเคชัน Lotus's SMART App จาก App Store หรือ Google Play Store
2. เปิดแอปพลิเคชัน Lotus's SMART App 
3. กดที่ "ลงชื่อเข้าใช้"
4. ใส่หมายเลขโทรศัพท์ที่สมัครสมาชิก My Lotus's 
5. ใส่รหัสผ่าน 
6. กดที่ "เข้าใช้งาน" ค่ะ หากมีคำถามเพิ่มเติม สามารถสอบถามน้องบัวได้เลยนะคะ ขอบคุณที่ใช้บริการค่ะ

In [17]:
user_query = "ถ้าต้องการสะสมคะแนนโลตัสต้องทำไงคะ"
response = await chat_engine.achat(user_query)
display(Markdown(str(response)))

2025-01-22 09:56:31,384 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/llm/v1/chat/completions "HTTP/1.1 200 OK"
2025-01-22 09:56:31,402 - INFO - Condensed question: วิธีการสะสมคะแนนโลตัสเมื่อใช้งานแอปพลิเคชัน Lotus's SMART App คืออะไร
2025-01-22 09:56:31,471 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/embedding/BAAI/bge-m3/embed "HTTP/1.1 200 OK"


Condensed question: วิธีการสะสมคะแนนโลตัสเมื่อใช้งานแอปพลิเคชัน Lotus's SMART App คืออะไร


2025-01-22 09:56:37,056 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/llm/v1/chat/completions "HTTP/1.1 200 OK"


คะแนนโลตัสหรือที่เรียกว่า Lotus Coins นั้น คุณสามารถสะสมได้จากการซื้อสินค้าที่ร่วมรายการค่ะ โดยทุกๆ 50 บาทที่คุณใช้จ่าย คุณจะได้รับ 0.5% ของจำนวนเงินที่ใช้จ่ายเป็น Lotus Coins ค่ะ เช่น ถ้าคุณซื้อสินค้า 1,000 บาท คุณจะได้รับ 5 Lotus Coins ค่ะ และคุณสามารถใช้คะแนนเหล่านี้เป็นส่วนลดในการซื้อสินค้าหรือแลกเป็นของแถมอื่นๆ ผ่านแอปพลิเคชัน Lotus's SMART App ได้เลยค่ะ หากมีคำถามเพิ่มเติม สามารถสอบถามน้องบัวได้เลยนะคะ ขอบคุณที่ใช้บริการค่ะ

In [18]:
user_query = "ขอที่อยู่ของ โลตัส โกเฟรช โพไร่หวาน"
response = await chat_engine.achat(user_query)
display(Markdown(str(response)))

2025-01-22 09:56:37,929 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/llm/v1/chat/completions "HTTP/1.1 200 OK"
2025-01-22 09:56:37,954 - INFO - Condensed question: ขอทราบที่อยู่ของสาขาโลตัส โกเฟรช โพไร่หวาน
2025-01-22 09:56:37,996 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/embedding/BAAI/bge-m3/embed "HTTP/1.1 200 OK"


Condensed question: ขอทราบที่อยู่ของสาขาโลตัส โกเฟรช โพไร่หวาน


2025-01-22 09:56:41,724 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/llm/v1/chat/completions "HTTP/1.1 200 OK"


ที่อยู่ของ โลตัส โกเฟรช โพไร่หวาน คือ เลขที่ 223 หมู่ 2 ต.โพไร่หวาน อ.เมืองเพชรบุรี จ.เพชรบุรี 76000 ค่ะ โทรศัพท์: 064-587-4193 ค่ะ หากมีคำถามเพิ่มเติม สามารถสอบถามน้องบัวได้เลยนะคะ ขอบคุณที่ใช้บริการค่ะ

In [19]:
user_query = "ข้อข้อมูลสาขาในจังหวัดบุรีรัมย์"
response = await chat_engine.achat(user_query)
display(Markdown(str(response)))

2025-01-22 09:56:42,534 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/llm/v1/chat/completions "HTTP/1.1 200 OK"
2025-01-22 09:56:42,554 - INFO - Condensed question: ขอข้อมูลเกี่ยวกับสาขาโลตัสในจังหวัดบุรีรัมย์
2025-01-22 09:56:42,607 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/embedding/BAAI/bge-m3/embed "HTTP/1.1 200 OK"


Condensed question: ขอข้อมูลเกี่ยวกับสาขาโลตัสในจังหวัดบุรีรัมย์


2025-01-22 09:56:57,732 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/llm/v1/chat/completions "HTTP/1.1 200 OK"


ข้อมูลสาขาโลตัสในจังหวัดบุรีรัมย์ มีดังนี้ค่ะ

1. โลตัส สาขาใหญ่ นางรอง 
- ที่อยู่: เลขที่ 904/7 ถ.โชคชัย-เดชอุดม ต.นางรอง อ.นางรอง จ.บุรีรัมย์ 31110
- โทรศัพท์: 04-463-2365
- เวลาเปิด-ปิด: 08:00-21:00 น.

2. โลตัส โกเฟรช โรงพยาบาลบุรีรัมย์ 
- ที่อยู่: เลขที่ 142/105-106,142/107 ถนนนิวาศ ตำบลในเมือง อำเภอเมืองบุรีรัมย์ จังหวัดบุรีรัมย์ 31000
- โทรศัพท์: 096-958-5878
- เวลาเปิด-ปิด: 24 ชั่วโมง

3. โลตัส โกเฟรช นางรอง 2 บุรีรัมย์ 
- ที่อยู่: เลขที่ 589/2,589/3 ถนนประจันตเขต ตำบลนางรอง อำเภอนางรอง จังหวัดบุรีรัมย์ 31110
- โทรศัพท์: 062-605-8841
- เวลาเปิด-ปิด: 06:00-23:00 น.

4. โลตัส โกเฟรช นางรอง บุรีรัมย์ 
- ที่อยู่: เลขที่ 443 ถนนโชคชัย-เดชอุดม ตำบลนางรอง อำเภอนางรอง จังหวัดบุรีรัมย์ 31110
- โทรศัพท์: 062-605-8185
- เวลาเปิด-ปิด: 06:00-23:00 น.

5. โลตัส สาขาใหญ่ ประโคนชัย 
- ที่อยู่: เลขที่ 420 หมู่ 8 ต.ประโคนชัย อ.ประโคนชัย จ.บุรีรัมย์ 31140
- โทรศัพท์: 04-467-0801
- เวลาเปิด-ปิด: 08:00-21:00 น.

หากมีคำถามเพิ่มเติม สามารถสอบถามน้องบัวได้เลยนะคะ ขอบคุณที่ใช้บริการค่ะ